# 0 · Explore and train a baseline

**The project.** Map land cover from Sentinel-2 images. Each image has a label picture saying what is on the ground at every pixel. We open the data, check it, and train a small baseline with Lightning.

```text
data/dynamic_world/
├── train/  images/<id>.tif   labels/<id>.tif
├── val/    ...
└── test/   ...
```

300 tiles of expert-annotated [Dynamic World training](https://doi.org/10.1594/PANGAEA.933475) labels, published as [fatmur/dynamicworld-300](https://huggingface.co/datasets/fatmur/dynamicworld-300). Notebook 1 improves the model, 2 rebuilds the inputs from STAC, 3 trains and publishes maps from that catalog.

**Data and interpretation.** These are expert-annotated training labels from [Tait et al. (2021)](https://doi.org/10.1594/PANGAEA.933475), paired with Sentinel-2 L2A imagery in the [teaching subset](https://huggingface.co/datasets/fatmur/dynamicworld-300). They have ten classes, including cloud, and nodata 255. They are not the automated nine-class Dynamic World probability product.

Keep tiles in their original train/validation/test splits before making chips. Nearby or overlapping tiles can still leak spatial information: inspect footprints before claiming geographic generalization. These short runs are workflow checks, not evidence of mapping accuracy.


**Run:** use the repository `.venv` kernel with `examples/` as the working directory. Requires `uv sync` and a ~400 MB download; CPU training is supported.

## Get the teaching data

Download the dataset (about 400 MB; running again only checks for changes).

In [ ]:
from pathlib import Path

import lightning as L
from huggingface_hub import snapshot_download

# One seed for the whole notebook, so the model's first weights and the shuffled
# batches repeat from run to run.
L.seed_everything(0)

root = Path(
    snapshot_download(
        "fatmur/dynamicworld-300",
        repo_type="dataset",
        revision="5e65026ad47e1f02f9a48a3efee1028db3bb0d1a",
        local_dir="data/dynamic_world",
    )
)

## Open an image and its labels

`gs.open_raster` returns a plain `xarray.Dataset`, one variable per band. The image file carries its RGB band choice; the label file carries its class names. `classes.yaml` adds the class colours with `.gs.write_flags`, so `.gs.plot()` needs no settings.

Start with eight prepared training pairs for this short run.

In [ ]:
import yaml
from IPython.display import display

import geosave as gs

classes = yaml.safe_load((root / "classes.yaml").read_text())

images = sorted((root / "train/images").glob("*.tif"))[:8]
labels = sorted((root / "train/labels").glob("*.tif"))[:8]

image = gs.open_raster(images[0])
label = gs.open_raster(labels[0])
label = label.gs.write_flags(classes["class_map"], classes["color_map"])

display(image)
image.gs.plot(xlabel="") + label.gs.plot(xlabel="")

## Explore

In [ ]:
import pandas as pd

summaries = []
for path in images[:5]:
    with gs.open_raster(path) as scene:
        reflectance = scene.gs.mask_and_scale()
        summaries.append(reflectance.gs.statistics())

pd.concat(summaries, keys=range(5), names=["tile", "band"])

Inspect the reflectance ranges and valid-pixel coverage above before training. Below is the mean class percentage across five example tiles (each tile gets equal weight; this is not a full-dataset census):

In [ ]:
import matplotlib.pyplot as plt

percentages = []
for path in labels[:5]:
    with gs.open_raster(path) as scene:
        flagged = scene["label"].gs.write_flags(classes["class_map"])
        percentages.append(flagged.gs.class_percent())

percent = pd.concat(percentages, axis=1).mean(axis=1)

percent.plot.bar(color=list(classes["color_map"].values()))
plt.ylabel("Mean percent per tile")
plt.title("Class balance in five training tiles")

Check which classes dominate your selected tiles and which are rare. Report mIoU next to accuracy, since accuracy alone rewards the common classes.

## Turn one pair into tensors

These prepared tiles already have matching image and label grids. Select the four image bands and decode reflectance. Labels are integer class IDs, so convert them directly without scaling. The image tensor has shape `(bands, height, width)` and the label tensor has shape `(height, width)`.

The selected training tiles have complete imagery. Label value 255 means unlabelled; the loss will ignore it.

In [ ]:
reflectance = image[["B02", "B03", "B04", "B08"]].gs.mask_and_scale()
image_tensor = reflectance.gs.to_tensor(dtype="float32")

label_tensor = label["label"].gs.to_tensor(dtype="int64")
image_tensor.shape, label_tensor.shape

## Read a pair when training asks for it

The Dataset reads the prepared pairs and converts them to tensors. Image values become reflectance; label IDs stay unchanged. Data acquisition and preparation are separate work, introduced in notebook 2.

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset


class DynamicWorld(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        with gs.open_raster(self.images[index], mask_and_scale=True) as scene:
            image = scene[["B02", "B03", "B04", "B08"]]
            image = image.gs.to_tensor(dtype="float32")

        with gs.open_raster(self.labels[index]) as scene:
            label = scene["label"].gs.to_tensor(dtype="int64")

        return image, label

## Keep the training and validation splits separate

Use eight prepared training pairs and two validation pairs. Training shuffles the pairs. Validation keeps their order so we can find the corresponding image and label when reviewing a prediction.

In [ ]:
val_images = sorted((root / "val/images").glob("*.tif"))[:2]
val_labels = sorted((root / "val/labels").glob("*.tif"))[:2]

train = DynamicWorld(images, labels)
val = DynamicWorld(val_images, val_labels)

train_loader = DataLoader(train, batch_size=8, shuffle=True)
val_loader = DataLoader(val, batch_size=8)

## Build a small segmentation model

An encoder turns pixels into features, a decoder brings them back to full size, and a head scores each of the 10 classes.

In [ ]:
from torch import nn


class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(4, 16, 3, stride=2, padding=1),
            nn.ReLU(),
        )

    def forward(self, image):
        return self.net(image)  # (B, 4, H, W) -> (B, 16, H/2, W/2)

The decoder brings the feature map back to the image size.

In [ ]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(16, 16, 3, padding=1),
            nn.ReLU(),
        )

    def forward(self, features):
        return self.net(features)  # (B, 16, H/2, W/2) -> (B, 16, H, W)

The head gives each pixel one score per class. Connect the three parts in their execution order.

In [ ]:
class Head(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Conv2d(16, 10, 1)

    def forward(self, features):
        return self.net(features)  # (B, 16, H, W) -> (B, 10, H, W)

In [ ]:
model = nn.Sequential(Encoder(), Decoder(), Head())
model

## Define the training behavior

Plain Lightning. Class 255 is ignored by the loss and metrics; mIoU is the usual segmentation score.

In [ ]:
import lightning as L
from torchmetrics import MetricCollection
from torchmetrics.classification import MulticlassAccuracy, MulticlassJaccardIndex


class Segmenter(L.LightningModule):
    def __init__(self, model):
        super().__init__()

        self.model = model

        self.val_metrics = MetricCollection(
            {
                "miou": MulticlassJaccardIndex(num_classes=10, ignore_index=255),
                "accuracy": MulticlassAccuracy(
                    num_classes=10, ignore_index=255, average="micro"
                ),
            },
            prefix="val_",
        )

        self.criterion = nn.CrossEntropyLoss(ignore_index=255)

    def forward(self, image):
        return self.model(image)

    def training_step(self, batch, batch_index):
        image, label = batch  # (B, C, H, W), (B, H, W)
        logits = self(image)  # (B, classes, H, W)
        loss = self.criterion(logits, label)

        self.log("train_loss", loss, on_step=True, on_epoch=True)

        return loss

    def validation_step(self, batch, batch_index):
        image, label = batch  # (B, C, H, W), (B, H, W)
        logits = self(image)  # (B, classes, H, W)
        loss = self.criterion(logits, label)

        self.log("val_loss", loss)
        self.val_metrics(logits, label)
        self.log_dict(self.val_metrics)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=1e-2)


module = Segmenter(model)

## Train a baseline

Run six short epochs on eight training pairs and two validation pairs. This small, prepared selection keeps the lesson quick; use a larger prepared dataset for a longer experiment. `trainer.fit` below starts a fresh run; to resume deliberately, pass a checkpoint path with `ckpt_path`. Change the logger version to keep separate experiments.

In [ ]:
import warnings

from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

# The tiles are small and few; Lightning's hint to add loader workers is noise here.
warnings.filterwarnings("ignore", ".*does not have many workers.*")

# Each notebook logs to a folder of its own; version 0 is its one run.
logger = TensorBoardLogger("lightning_logs", name="00_basics", version=0)
# Save the last epoch for an explicit future resume.
checkpoint = ModelCheckpoint(save_last=True)

trainer = L.Trainer(
    max_epochs=6,
    log_every_n_steps=1,
    logger=logger,
    callbacks=[checkpoint],
)
trainer.fit(module, train_loader, val_loader)

Compare `train_loss_step` with `val_loss` for generalisation, and `val_miou` and `val_accuracy` per epoch.

### Optional: inspect training logs

Run this in a separate cell when you want TensorBoard. It starts a local service.

```python
%load_ext tensorboard
%tensorboard --logdir lightning_logs
```

## Predict from an image

Inference needs an image and the trained model. Give the prediction the image's coordinates to place it on the map, and `classes.yaml` for class names and colours. Reference labels are not part of this path.

In [ ]:
import xarray as xr

image = gs.open_raster(val_images[0])

reflectance = image[["B02", "B03", "B04", "B08"]].gs.mask_and_scale()
pixels = reflectance.gs.to_tensor(dtype="float32")
batch = pixels.unsqueeze(0).to(module.device)

module.eval()
with torch.inference_mode():
    logits = module(batch)
    prediction = logits.argmax(dim=1)[0].cpu().numpy().astype("uint8")

predicted = xr.DataArray(prediction, coords=image.coords, dims=("y", "x"), name="label")
predicted = predicted.gs.write_nodata(classes["ignore_index"])
predicted = predicted.gs.write_flags(classes["class_map"], classes["color_map"])

image.gs.plot(xlabel="") + predicted.gs.plot(xlabel="")

### Compare with a reference when one is available

This validation image happens to have a reference label. Open it separately to inspect the model's errors.

In [ ]:
reference = gs.open_raster(val_labels[0])["label"]
reference = reference.gs.write_flags(classes["class_map"], classes["color_map"])

reference.gs.plot(title="Reference") + predicted.gs.plot(title="Prediction")

## What happened

We opened satellite files as xarray Datasets, summarised them with `.gs.statistics()` and `.gs.class_percent()`, turned bands into tensors, trained with Lightning, and put a prediction back on the tile's coordinates with its class names and colours. Six epochs only show the loop works; expect poor predictions.

| Next | Question |
| --- | --- |
| 1 · Model | How do I grow this model, swap its parts and compare runs? |
| 2 · Build a dataset | How do I build a dataset like this for my own labels, places and dates? |
| 3 · Train and predict | How do I train on my own dataset and map an unlabelled place? |

## More things you can do with this baseline (optional)

The baseline workflow is complete. You can skip this section and continue to [notebook 1](01_model.ipynb).

### Decide which classes need a closer look

Suppose the overall score looks reasonable, but trees matter most to your project. Use GeoSave's `MulticlassMetrics` to see IoU for each class, then inspect examples of the classes your model struggles with before deciding whether to collect more labels or change the model.

This uses the same two validation tiles. A class missing from these tiles cannot tell you how well the model recognizes it; check the label maps and their `.gs.class_percent()` before interpreting a low score.

In [ ]:
from geosave.ml.metrics import MulticlassMetrics

metrics = MulticlassMetrics(
    num_classes=len(classes["class_map"]),
    ignore_index=classes["ignore_index"],
    labels=classes["class_map"],
    metrics=["iou.per_class"],
)

module.eval()
with torch.inference_mode():
    for image, target in val_loader:
        image = image.to(module.device)
        logits = module(image).cpu()

        metrics.update(logits, target)

metrics.compute()

### Make a review sheet for a collaborator

Use the prediction and reference comparison above on another validation image. Put the input image beside both maps so a reviewer can distinguish a model mistake from an ambiguous label. The prediction still uses only the image and model; the reference is for review.

### Compare a change fairly

Save this run's settings and keep the same validation tiles when changing the model or augmentation. [Notebook 1](01_model.ipynb) continues that experiment. Keep the test split for evaluating your final choice.